In [2]:
from pathlib import Path
import sys

# Project root
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\hp\Documents\Anemia_Fusion_Net_project


In [4]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*60)
print("Device :", device)
print("="*60)

Device : cpu


In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print("PROJECT ROOT:", PROJECT_ROOT)

PROJECT ROOT: c:\Users\hp\Documents\Anemia_Fusion_Net_project


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split

from ml.datasets.image_dataset import ImageDataset
from ml.models.cnn_model import CNNModel

from ml.config import *

PROJECT_ROOT : C:\Users\hp\Documents\Anemia_Fusion_Net_project
TRAIN_CSV    : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\train.csv
VAL_CSV      : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\valid.csv
IMAGE_DIR    : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\eye_image
Device       : cpu


In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

for file in PROJECT_ROOT.rglob("config.py"):
    print(file)

c:\Users\hp\Documents\Anemia_Fusion_Net_project
c:\Users\hp\Documents\Anemia_Fusion_Net_project\ml\config.py


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Image Model Training")
print("=" * 70)

print("Project Root :", PROJECT_ROOT)
print("Device       :", device)

print("\nConfiguration")
print("-" * 70)

print("TRAIN_CSV :", TRAIN_CSV)
print("VAL_CSV   :", VAL_CSV)
print("IMAGE_DIR :", IMAGE_DIR)

print("=" * 70)

Image Model Training
Project Root : C:\Users\hp\Documents\Anemia_Fusion_Net_project
Device       : cpu

Configuration
----------------------------------------------------------------------
TRAIN_CSV : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\train.csv
VAL_CSV   : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\valid.csv
IMAGE_DIR : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\eye_image


In [17]:
dataset = ImageDataset(
    csv_file=TRAIN_CSV,
    image_root=IMAGE_DIR
)

print("=" * 70)
print("Image Dataset Loaded Successfully")
print("=" * 70)

print("Total Samples :", len(dataset))

Image Dataset Loaded Successfully
Total Samples : 603


In [16]:
import inspect

from ml.datasets.image_dataset import ImageDataset

print(inspect.signature(ImageDataset.__init__))

(self, csv_file, image_root, transform=None)


In [18]:
image, label = dataset[0]

print("=" * 70)
print("Sample Loaded")
print("=" * 70)

print("Image Shape :", image.shape)
print("Label :", label)

Sample Loaded
Image Shape : torch.Size([3, 224, 224])
Label : 0


In [19]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("=" * 70)
print("Dataset Split Successfully")
print("=" * 70)
print("Training Samples  :", len(train_dataset))
print("Validation Samples:", len(val_dataset))

Dataset Split Successfully
Training Samples  : 482
Validation Samples: 121


In [20]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("=" * 70)
print("DataLoaders Created Successfully")
print("=" * 70)
print("Train Batches :", len(train_loader))
print("Validation Batches :", len(val_loader))

DataLoaders Created Successfully
Train Batches : 16
Validation Batches : 4


In [21]:
images, labels = next(iter(train_loader))

print("=" * 70)
print("One Batch Loaded Successfully")
print("=" * 70)

print("Images Shape :", images.shape)
print("Labels Shape :", labels.shape)
print("Labels :", labels)

One Batch Loaded Successfully
Images Shape : torch.Size([32, 3, 224, 224])
Labels Shape : torch.Size([32])
Labels : tensor([0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 1, 1, 1, 0, 1, 0])


In [22]:
model = CNNModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

print("=" * 70)
print("CNN Model Loaded Successfully")
print("=" * 70)

print("Loss Function :", criterion)
print("Optimizer     :", optimizer.__class__.__name__)
print("Scheduler     :", scheduler.__class__.__name__)

CNN Model Loaded Successfully
Loss Function : CrossEntropyLoss()
Optimizer     : AdamW
Scheduler     : CosineAnnealingLR


In [25]:
images = images.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

print("=" * 70)
print("Forward Pass Successful")
print("=" * 70)

print("Input Shape  :", images.shape)
print("Output Shape :", outputs.shape)

Forward Pass Successful
Input Shape  : torch.Size([32, 3, 224, 224])
Output Shape : torch.Size([32, 2])


In [26]:
loss = criterion(outputs, labels)

print("=" * 70)
print("Loss")
print("=" * 70)

print("Loss :", loss.item())

Loss
Loss : 0.6948372721672058


In [27]:
probabilities = torch.softmax(outputs, dim=1)

predictions = torch.argmax(outputs, dim=1)

print("=" * 70)
print("Predictions")
print("=" * 70)

print(predictions)

Predictions
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 1])


In [28]:
print("=" * 70)
print("Notebook 10 Completed Successfully")
print("=" * 70)

print("✓ Image Dataset Loaded")
print("✓ DataLoader Created")
print("✓ CNN Model Loaded")
print("✓ Forward Pass Verified")
print("✓ Loss Computed")
print("✓ Predictions Generated")

print("\nNext Step:")
print("Notebook 11 - Clinical Model Training")

Notebook 10 Completed Successfully
✓ Image Dataset Loaded
✓ DataLoader Created
✓ CNN Model Loaded
✓ Forward Pass Verified
✓ Loss Computed
✓ Predictions Generated

Next Step:
Notebook 11 - Clinical Model Training
